## ML Pipelines for Apache Spark


## Transformers and Estimators

Before building a pipeline, it helps to be precise about the two kinds of objects it can contain.

**Transformer** — has only a `transform(df)` method. It applies a fixed, data-independent operation: `VectorAssembler`, `OneHotEncoder` (after fitting), `Tokenizer`. No learning happens; the same logic applies to any DataFrame.

**Estimator** — has a `fit(df)` method that *learns* something from the data and returns a fitted **transformer** (a `Model`). Examples: `StringIndexer`, `Imputer`, `MinMaxScaler`, `Word2Vec`, `LogisticRegression`. You must call `.fit()` before you can call `.transform()`.

The distinction matters because a `Pipeline` handles them differently under the hood: for each stage it calls `fit()` if the stage is an estimator, `transform()` if it is a transformer, passing the output of one stage as the input to the next.

### Data contamination

Estimators *learn from whatever data you pass to `.fit()`*. If you fit an `Imputer` or `MinMaxScaler` on the full dataset before splitting, the learned statistics (mean, min, max) will reflect test-set values too — the model has indirectly 'seen' the test set before evaluation. This is **data leakage**, and it produces optimistically biased metrics.

The correct discipline is: **fit only on training data, transform both splits**.

```python
# Wrong — leaks test statistics into the fitted parameters
imputer_model = imputer.fit(full_dataset)

# Correct — learns only from training data
imputer_model = imputer.fit(training)
train_imputed = imputer_model.transform(training)
test_imputed  = imputer_model.transform(test)
```

### Is there a `fit_transform()` like in scikit-learn?

No. Spark ML does not have `fit_transform()`, and the `Pipeline` object is the idiomatic replacement. Calling `pipeline.fit(training)` fits every estimator stage on the training data, producing a `PipelineModel`. That model is then a pure transformer: `pipeline_model.transform(test)` applies all stages — with the parameters learned from training — to the test set without refitting anything.

## Building a Pipeline

A `Pipeline` is itself an estimator whose only parameter is `stages`: an ordered list of transformers and estimators. Calling `.fit()` on a Pipeline produces a `PipelineModel`, which is a transformer. That `PipelineModel` can then be applied to new data with a single `.transform()` call — the entire preprocessing chain runs automatically in the correct order.

This is the key practical advantage over Part 1's manual approach: the pipeline guarantees that test data and any future production data go through exactly the same transformations as the training data, fitted on the training data only.

A comprehensive, general machine learning pipeline typically consists of several stages. These stages may vary depending on the specific problem, dataset, and chosen algorithm, but the following outline serves as a good starting point:

##### 1. Define the problem and objectives:
Clearly state the problem you aim to solve, the desired outcome, and any specific constraints or requirements.

##### 2. Collect and preprocess the data:

##### 2.1 Data acquisition:
Gather data from various sources such as databases, APIs, web scraping, or manual input.

##### 2.2 Data cleaning:
Handle missing, inconsistent, or duplicate data.

###### 2.3 Data transformation:
Convert data into a suitable format for analysis, including encoding categorical variables and normalizing numerical features.

##### 3. Explore and analyze the data:

###### 3.1 Descriptive statistics:
Analyze summary statistics to better understand the data's distribution and relationships.

###### 3.2 Data visualization:
Use plots and graphs to identify patterns, trends, and outliers in the data.

##### 4. Feature engineering and selection:

###### 4.1 Feature engineering:
Create new features from existing data to improve model performance.
###### 4.2 Feature selection:
Choose a subset of features that are most relevant to the problem, minimizing noise and redundancy.
###### 4.3 Split the data into training and testing sets:
Divide the dataset into separate subsets for training and evaluation to prevent overfitting and assess model performance.

##### 5. Train the model:
Select an appropriate machine learning algorithm based on the problem type (classification, regression, clustering, etc.). Train the model using the training dataset and fine-tune its parameters to optimize performance.

##### 6. Evaluate the model:
Assess the model's performance using evaluation metrics such as accuracy, precision, recall, F1-score, or mean squared error, depending on the problem type. Evaluate the model on the testing dataset to ensure it generalizes well to unseen data.

##### 7. Model selection and hyperparameter tuning:
If multiple models are being considered, compare their performance and choose the best one. Optimize the chosen model by tuning its hyperparameters using techniques like grid search, random search, or Bayesian optimization.

##### 8. Interpret and validate the model:
Analyze the model's results to understand the underlying patterns and relationships it has learned. Perform cross-validation or use holdout datasets to further validate its performance.

##### 9. Deploy the model:
Integrate the trained model into a production environment or application, enabling it to make predictions on new, unseen data.

##### 10. Monitor and maintain the model:
Regularly evaluate the model's performance to ensure it remains accurate and relevant. Update the model with new data, retrain, and redeploy as needed.

Remember, this is a general outline, and specific steps may vary depending on the project's requirements and the chosen machine learning algorithm.

##### Some of the advantages of using pipelines:

* **Code Simplification:** A pipeline encapsulates multiple stages of data preprocessing, feature extraction, and machine learning model training into a consistent and streamlined API. This helps to make the code more readable and organized.

* **Reuse and Sharing:** Once a pipeline is defined, it can be reused across different contexts and shared among different projects. This can significantly speed up development and decrease the chance of errors.

* **Reproducibility:** A pipeline provides a recipe for data transformations and training steps. This helps ensure that the same steps are followed every time the pipeline is run, making results reproducible.

* **Efficiency:** The pipeline API helps avoid intermediate storage of transformed datasets by fusing multiple stages. This makes pipelines more efficient by reducing data shuffling and memory usage.

* **Model Selection and Hyperparameter Tuning:** With a pipeline, you can perform model selection and hyperparameter tuning in a more streamlined and efficient manner. The CrossValidator and TrainValidationSplit classes in PySpark allow you to search the hyperparameter space of an entire pipeline, which can be more effective than tuning the stages independently.

* **Pipeline Persistence:** PySpark also provides the ability to save and load pipelines. This allows you to store the pipeline structure separate from the data, so you can reuse the same pipeline with new data, or continue from where you left off.

* **Error Handling:** If an error occurs at any stage, PySpark stops the execution of the pipeline. This way, you won't end up with a partially processed dataset that could lead to incorrect conclusions or models.

In [1]:
# Import necessary libraries
from pyspark.sql.functions import * # Usually a bad idea
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, Normalizer, SQLTransformer
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
# from sklearn.metrics import auc
import pandas as pd

In [2]:
# #Checking the installed Java version
# !java -version
# !pip install pyspark 
# # Install Java 17
# !sudo apt-get update
# !sudo apt-get install -y openjdk-17-jdk-headless

# !java -version

# Set JAVA_HOME to Java 17
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession

spark = SparkSession.builder\
        .master("local[*]")\
        .appName("ML") \
        .getOrCreate()
print("Spark ready:", spark.version)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/13 13:09:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark ready: 4.1.1


In [3]:
# Load the dataset into a Spark DataFrame
file_location       = "/teamspace/studios/this_studio/week05/ifood_dataset_lab003.csv" 
# Import data
customers_ifood_raw = spark.read.load(file_location,
                     format      = "csv",           
                     sep         = ";",           
                     inferSchema = "true",        
                     header      = "true")

In [4]:
customers_ifood_raw.show(5, truncate = True)

26/05/13 13:09:07 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----+----------+----------+--------------+------+-------+--------+-----------+-------+--------+---------+---------------+---------------+----------------+------------+-----------------+---------------+-------------------+-----------------+-----------------+------------+------------+------------+------------+------------+--------+-------------+---------+--------+
|  ID|Year_Birth| Education|Marital_Status|Income|Kidhome|Teenhome|Dt_Customer|Recency|MntWines|MntFruits|MntMeatProducts|MntFishProducts|MntSweetProducts|MntGoldProds|NumDealsPurchases|NumWebPurchases|NumCatalogPurchases|NumStorePurchases|NumWebVisitsMonth|AcceptedCmp3|AcceptedCmp4|AcceptedCmp5|AcceptedCmp1|AcceptedCmp2|Complain|Z_CostContact|Z_Revenue|Response|
+----+----------+----------+--------------+------+-------+--------+-----------+-------+--------+---------+---------------+---------------+----------------+------------+-----------------+---------------+-------------------+-----------------+-----------------+----------

In [5]:
# Check for missing values in the dataframe
customers_ifood_raw.select([count(when(col(c).isNull(), c)).alias(c) for c in customers_ifood_raw.columns]).show()

+---+----------+---------+--------------+------+-------+--------+-----------+-------+--------+---------+---------------+---------------+----------------+------------+-----------------+---------------+-------------------+-----------------+-----------------+------------+------------+------------+------------+------------+--------+-------------+---------+--------+
| ID|Year_Birth|Education|Marital_Status|Income|Kidhome|Teenhome|Dt_Customer|Recency|MntWines|MntFruits|MntMeatProducts|MntFishProducts|MntSweetProducts|MntGoldProds|NumDealsPurchases|NumWebPurchases|NumCatalogPurchases|NumStorePurchases|NumWebVisitsMonth|AcceptedCmp3|AcceptedCmp4|AcceptedCmp5|AcceptedCmp1|AcceptedCmp2|Complain|Z_CostContact|Z_Revenue|Response|
+---+----------+---------+--------------+------+-------+--------+-----------+-------+--------+---------+---------------+---------------+----------------+------------+-----------------+---------------+-------------------+-----------------+-----------------+------------+---

This is the same dataset we used last week. Last week we saw we needed to clean this data in various ways. We are going to do so here using transformers so that this data cleaning can become part of our pipeline.

In [6]:
# First we are going to create a small sample pipeline to test our transformers as we go
test_df = customers_ifood_raw.sample(0.01)
test_df.show()

+-----+----------+----------+--------------+------+-------+--------+-----------+-------+--------+---------+---------------+---------------+----------------+------------+-----------------+---------------+-------------------+-----------------+-----------------+------------+------------+------------+------------+------------+--------+-------------+---------+--------+
|   ID|Year_Birth| Education|Marital_Status|Income|Kidhome|Teenhome|Dt_Customer|Recency|MntWines|MntFruits|MntMeatProducts|MntFishProducts|MntSweetProducts|MntGoldProds|NumDealsPurchases|NumWebPurchases|NumCatalogPurchases|NumStorePurchases|NumWebVisitsMonth|AcceptedCmp3|AcceptedCmp4|AcceptedCmp5|AcceptedCmp1|AcceptedCmp2|Complain|Z_CostContact|Z_Revenue|Response|
+-----+----------+----------+--------------+------+-------+--------+-----------+-------+--------+---------+---------------+---------------+----------------+------------+-----------------+---------------+-------------------+-----------------+-----------------+-------

We were using SQL queries for data transformation (with the spark dataframe syntax). We can use SQLTransformer for this instead.

https://spark.apache.org/docs/latest/ml-features.html#sqltransformer

In [7]:
# 1. Preprocessing
## 1.1. Drop the nans
# Below is the code from last week. Now we use the SQLTransformer to achieve the same result.
# customers_ifood = customers_ifood_raw.na.drop()
null_transformer = SQLTransformer(statement = "SELECT * FROM __THIS__ WHERE Income IS NOT NULL")
test_df = null_transformer.transform(test_df)
test_df.show()

+-----+----------+----------+--------------+------+-------+--------+-----------+-------+--------+---------+---------------+---------------+----------------+------------+-----------------+---------------+-------------------+-----------------+-----------------+------------+------------+------------+------------+------------+--------+-------------+---------+--------+
|   ID|Year_Birth| Education|Marital_Status|Income|Kidhome|Teenhome|Dt_Customer|Recency|MntWines|MntFruits|MntMeatProducts|MntFishProducts|MntSweetProducts|MntGoldProds|NumDealsPurchases|NumWebPurchases|NumCatalogPurchases|NumStorePurchases|NumWebVisitsMonth|AcceptedCmp3|AcceptedCmp4|AcceptedCmp5|AcceptedCmp1|AcceptedCmp2|Complain|Z_CostContact|Z_Revenue|Response|
+-----+----------+----------+--------------+------+-------+--------+-----------+-------+--------+---------+---------------+---------------+----------------+------------+-----------------+---------------+-------------------+-----------------+-----------------+-------

In [8]:

# 2. Feature engineering
## 2.1. Derive age from birth year, i.e. transform year of birth into age
# Below is the code from last week. Use the SQLTransformer to achieve the same result.
# customers_ifood = customers_ifood.withColumn("Age", 2026 - col("Year_Birth"))
age_column_transformer = SQLTransformer(statement = "SELECT *, 2026 - Year_Birth AS Age FROM __THIS__")
test_df = age_column_transformer.transform(test_df)

In [9]:
test_df.show(5, truncate = True)

+----+----------+----------+--------------+------+-------+--------+-----------+-------+--------+---------+---------------+---------------+----------------+------------+-----------------+---------------+-------------------+-----------------+-----------------+------------+------------+------------+------------+------------+--------+-------------+---------+--------+---+
|  ID|Year_Birth| Education|Marital_Status|Income|Kidhome|Teenhome|Dt_Customer|Recency|MntWines|MntFruits|MntMeatProducts|MntFishProducts|MntSweetProducts|MntGoldProds|NumDealsPurchases|NumWebPurchases|NumCatalogPurchases|NumStorePurchases|NumWebVisitsMonth|AcceptedCmp3|AcceptedCmp4|AcceptedCmp5|AcceptedCmp1|AcceptedCmp2|Complain|Z_CostContact|Z_Revenue|Response|Age|
+----+----------+----------+--------------+------+-------+--------+-----------+-------+--------+---------+---------------+---------------+----------------+------------+-----------------+---------------+-------------------+-----------------+-----------------+--

In [10]:
## 2.2. Calculate the seniority of the customer from the data the customer signed up
# Below is the code from last week. Use the SQLTransformer to achieve the same result. Use Years_Cust as the new name.

# customers_ifood = customers_ifood.withColumn("Dt_Customer", year(lit("2026")) - year("Dt_Customer"))

seniority_transformer = SQLTransformer(statement="SELECT *, 2026 - year(Dt_Customer) AS Years_Cust FROM __THIS__")

test_df = seniority_transformer.transform(test_df)
test_df.select('Id','Age','Dt_Customer','Years_Cust').show()

+-----+---+-----------+----------+
|   Id|Age|Dt_Customer|Years_Cust|
+-----+---+-----------+----------+
| 5342| 50| 2012-08-08|        14|
|  709| 74| 2012-12-09|        14|
| 9703| 56| 2012-10-15|        14|
| 6131| 46| 2013-12-01|        13|
|  178| 70| 2013-02-18|        13|
|  895| 78| 2012-12-09|        14|
|10479| 51| 2012-12-07|        14|
| 9984| 45| 2013-03-27|        13|
|10552| 47| 2013-05-20|        13|
| 6749| 60| 2012-08-08|        14|
| 4136| 34| 2012-12-03|        14|
| 6246| 73| 2013-11-13|        13|
| 3102| 45| 2013-10-16|        13|
| 5117| 63| 2012-08-02|        14|
+-----+---+-----------+----------+



In [11]:
## 2.3. Filter the DataFrame to keep only rows where the Marital_Status column does not contain values_to_drop
# Below is the code from last week. Use the SQLTransformer to achieve the same result.

# values_to_drop = ['YOLO', 'Absurd', 'Alone']
# customers_ifood = customers_ifood.filter(~col("Marital_Status").isin(values_to_drop))

filter_status_transformer = SQLTransformer(statement="SELECT * FROM __THIS__ WHERE Marital_Status NOT IN ('YOLO', 'Absurd', 'Alone')")
test_df = filter_status_transformer.transform(test_df)

We were already using transformers last week, let's setup the same transformers.

In [12]:
## 2.4 Convert Education and Marital_Status columns to numerical form by applying one-hot encoding to the indexed columns
indexer_edu = StringIndexer(inputCol='Education', outputCol='EducationIndex')     

indexer_mar = StringIndexer(inputCol='Marital_Status', outputCol='MaritalIndex') # TRANSFORMER

encoder = OneHotEncoder(inputCols=['EducationIndex', 'MaritalIndex'],            # TRANSFORMER 
                                 outputCols=['EducationVec', 'MaritalVec']) 

We can turn to pipelines to connect our transformers.

In [13]:
input_features_numeric = ['Income', 'Recency', 'NumWebVisitsMonth', 'MntWines', 'MntMeatProducts', 'MntGoldProds']
input_features_categorical = ['EducationVec', 'MaritalVec']
assembler_num = VectorAssembler(inputCols=input_features_numeric, outputCol="numeric_features")   # TRANSFORMER (INITIALIZATION)
normaliser_num = Normalizer(inputCol="numeric_features", outputCol="normalized_features") # TRANSFORMER (INITIALIZATION)
assembler = VectorAssembler(inputCols=["normalized_features", 'EducationVec', 'MaritalVec'], outputCol="features") 
assembler_normalized_features = VectorAssembler(inputCols=["normalized_features"], outputCol="features") 

In [14]:
# Test the pipeline so far
preproc_pipeline = Pipeline(stages = [null_transformer, seniority_transformer, age_column_transformer])
data_pipeline = Pipeline(stages=[preproc_pipeline, filter_status_transformer, indexer_edu, indexer_mar, encoder, assembler_num, normaliser_num, assembler])
test_df_pipeline = customers_ifood_raw.sample(0.01)
data_model = data_pipeline.fit(test_df_pipeline)

In [15]:
test_df_transformed = data_model.transform(test_df_pipeline)
test_df_transformed.select('Id','EducationVec','MaritalVec','numeric_features','normalized_features','features').show(5,truncate=False)

+-----+-------------+-------------+------------------------------------+--------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Id   |EducationVec |MaritalVec   |numeric_features                    |normalized_features                                                                                                             |features                                                                                                                                                        |
+-----+-------------+-------------+------------------------------------+--------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------

#### Let's move towards applying our ML model

In [16]:
# Train-test split
train_data, test_data = customers_ifood_raw.randomSplit([0.8, 0.2])

In [17]:
# Define the classifier
lr = LogisticRegression(featuresCol='features', labelCol='Response')     # TRANSFORMER  (INITIALIZATION)

# Create pipeline
lr_pipeline = Pipeline(stages=[data_pipeline,lr])

# Train the model
lr_model = lr_pipeline.fit(train_data)          # TRANSFORMER  (FIT)

26/05/13 13:09:15 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


In [18]:
# Make predictions
lr_predictions = lr_model.transform(test_data)   # TRANSFORMER(TRANSFORM)

# Evaluate the models
evaluator = BinaryClassificationEvaluator(labelCol='Response', metricName='areaUnderROC')

lr_auc = evaluator.evaluate(lr_predictions)

print("Logistic Regression AUC: ", lr_auc)

Logistic Regression AUC:  0.7841512469831077


In [19]:
lr_predictions.select('ID','response','prediction').show(50)

+----+--------+----------+
|  ID|response|prediction|
+----+--------+----------+
|  13|       0|       0.0|
|  17|       0|       0.0|
|  24|       0|       0.0|
|  25|       0|       0.0|
|  35|       1|       0.0|
|  78|       0|       0.0|
| 158|       1|       0.0|
| 178|       0|       0.0|
| 182|       0|       0.0|
| 199|       0|       0.0|
| 274|       0|       0.0|
| 368|       0|       0.0|
| 498|       0|       0.0|
| 520|       0|       0.0|
| 523|       0|       0.0|
| 524|       0|       0.0|
| 535|       0|       0.0|
| 550|       0|       0.0|
| 624|       1|       0.0|
| 635|       0|       0.0|
| 642|       0|       0.0|
| 701|       0|       0.0|
| 736|       0|       0.0|
| 749|       1|       0.0|
| 793|       0|       0.0|
| 796|       0|       0.0|
| 800|       0|       0.0|
| 810|       0|       0.0|
| 837|       0|       0.0|
| 843|       0|       0.0|
| 851|       0|       0.0|
| 940|       0|       0.0|
| 954|       0|       0.0|
| 977|       0|       0.0|
|

#### Feature Importances

In [20]:
input_features = input_features_numeric
input_features

['Income',
 'Recency',
 'NumWebVisitsMonth',
 'MntWines',
 'MntMeatProducts',
 'MntGoldProds']

In [22]:
# Extract the feature importances or coefficients
lr_coefficients = lr_model.stages[-1].coefficients[0:len(input_features)]

# Display the feature importances or coefficients
print("Logistic Regression coefficients:")
print(lr_coefficients)

Logistic Regression coefficients:
[ 1876.68392717 -1029.87257528  2258.17866918    77.84480369
   155.28993151    39.75027134]


In [23]:
# Extract the feature names from the pipeline
feature_names = assembler.getInputCols()

# Create a DataFrame to display feature importances or coefficients
lr_feature_df = pd.DataFrame({"feature": input_features, "coefficient": lr_coefficients})

# Display the top 10 most important features for each model
print("Top 10 important features for Logistic Regression:")
print(lr_feature_df.nlargest(10, "coefficient"))

Top 10 important features for Logistic Regression:
             feature  coefficient
2  NumWebVisitsMonth  2258.178669
0             Income  1876.683927
4    MntMeatProducts   155.289932
3           MntWines    77.844804
5       MntGoldProds    39.750271
1            Recency -1029.872575


## Saving and Loading a Pipeline

A fitted `PipelineModel` can be written to disk and reloaded later. This is how a trained model moves from a training notebook to a production scoring job — the loaded model is a fully functional transformer that applies all stages to new data with a single `.transform()` call, with no retraining.
pipeline_model.write().overwrite().save("./hmspia_pipeline_lr")

In [25]:
lr_model.write().overwrite().save("./hmspia_pipeline_lr")


from pyspark.ml import PipelineModel
loaded_model = PipelineModel.load("./hmspia_pipeline_lr")
auc_loaded = evaluator.evaluate(loaded_model.transform(test_data))
print(f"AUC from loaded pipeline: {auc_loaded:.4f}")

AUC from loaded pipeline: 0.7842


## Hyperparameter Tuning with CrossValidator

`CrossValidator` wraps a pipeline (or any estimator) and a grid of hyperparameter values, refitting the pipeline once for each combination across K folds of the training data. It returns the best model according to the evaluator metric.

`ParamGridBuilder` constructs the grid. Each `.addGrid(param, values)` call adds one hyperparameter with a list of candidate values; the builder takes the Cartesian product across all added parameters.

Note: Spark ML has no random grid search equivalent. `ParamGridBuilder` always evaluates the full Cartesian product.

In [28]:
# 1. Define the Preprocessing Pipeline
# This stage handles all feature engineering: indexing, encoding, imputing, and scaling.
# Nested pipelines allow for cleaner separation between feature prep and model training.
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, ClusteringEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.stat import Summarizer

# 2. Initialize the Estimator
# We use Logistic Regression as our base learner, targeting the 'TARGET' label.
lr_crossval = LogisticRegression(featuresCol="features", labelCol="Response")

# 3. Create the Full Cross-Validation Pipeline
# By nesting the preprocessing pipeline, we ensure that every fold of cross-validation 
# re-runs the preprocessing to prevent data leakage from the validation sets.
crossval_pipeline = Pipeline(stages=[data_pipeline, lr_crossval])

# 4. Construct the Parameter Grid
# We explore combinations of regularization (regParam) and the L1/L2 mix (elasticNetParam).
# .build() generates the cartesian product of these parameters.
param_grid = (
    ParamGridBuilder()
    .addGrid(lr_crossval.regParam,        [0.01, 0.1])
    .addGrid(lr_crossval.elasticNetParam, [0.0,  1.0])
    .build()
)

# 5. Configure the Cross-Validator
# Estimator: The pipeline to be tuned.
# numFolds: Splits the training data into 3 parts; rotates 1 for validation and 2 for training.
cross_validator = CrossValidator(
    estimator          = crossval_pipeline,
    estimatorParamMaps = param_grid,
    evaluator          = evaluator,
    numFolds           = 3,
    seed               = 753,
)

# 6. Fit and Evaluate
# This triggers (numFolds * numParamCombinations) training runs.
cross_validator_model = cross_validator.fit(train_data)
crossval_results      = cross_validator_model.transform(test_data)
auc_cv                = evaluator.evaluate(crossval_results)

print(f"Cross-validated AUC-ROC: {auc_cv:.4f}")
#print(f"Avg metrics per fold:    {[round(m, 4) for m in cross_validator_model.avgMetrics]}")


Cross-validated AUC-ROC: 0.7684


In [29]:
# The best model is accessible directly
best_pipeline = cross_validator_model.bestModel
best_lr       = best_pipeline.stages[-1]   # LR is last stage of inner pipeline
print(f"Best regParam:        {best_lr.getRegParam()}")
print(f"Best elasticNetParam: {best_lr.getElasticNetParam()}")

Best regParam:        0.01
Best elasticNetParam: 0.0


## Exercise 1
Does the model perform better when using only normalized numeric features to predict the target?

## Exercise 2
How does keeping unusual values in the marital status variable affect model performance?

# Exercise 3
Build a crossvalidor to find the best randomforest model.
